In [1]:
import numpy as np
import pandas as pd
from pandas import Series, DataFrame

df = pd.read_csv('../../data/train.csv')

# LoanNr_ChkDgt 	
Text 	Unique loan application identifier. 	Useful for tracking or deduplication checks.

In [2]:
# LoanNr_ChkDgt audit: counts, percentages, missing values, invalid format, and duplicates
loan_raw = df['LoanNr_ChkDgt']

# Standardize to string for consistent checks
loan_std = loan_raw.astype('string').str.strip()

# Missing includes true NaN and blank strings
missing_mask_loan = loan_raw.isna() | loan_std.isna() | (loan_std == '')

# Valid loan number format: exactly 10 digits
valid_mask_loan = loan_std.str.fullmatch(r'\d{10}', na=False)
invalid_mask_loan = (~missing_mask_loan) & (~valid_mask_loan)

# Duplicate check over non-missing standardized values
duplicate_mask_loan = (~missing_mask_loan) & loan_std.duplicated(keep=False)

total_rows = len(df)
summary_loan = pd.DataFrame(
    {'count': [
        total_rows,
        int(missing_mask_loan.sum()),
        int(invalid_mask_loan.sum()),
        int(valid_mask_loan.sum()),
        int(duplicate_mask_loan.sum()),
        int(loan_std[duplicate_mask_loan].nunique(dropna=True))
    ]},
    index=[
        'total_rows',
        'missing',
        'invalid_not_10_digits',
        'valid_10_digits',
        'duplicate_rows',
        'duplicate_unique_values'
    ]
)
summary_loan['percentage_%'] = (summary_loan['count'] / total_rows * 100).round(2)

print('LoanNr_ChkDgt data quality summary:')
display(summary_loan)

# Full distribution (including missing shown as MISSING)
loan_distribution = (
    loan_std.fillna('MISSING')
    .replace({'': 'MISSING'})
    .value_counts(dropna=False)
    .rename_axis('LoanNr_ChkDgt_value')
    .to_frame('count')
)
loan_distribution['percentage_%'] = (loan_distribution['count'] / total_rows * 100).round(2)

print('\nLoanNr_ChkDgt value distribution (all categories):')
display(loan_distribution.head(30))

# Invalid category breakdown
invalid_loan_categories = (
    loan_std[invalid_mask_loan]
    .value_counts(dropna=False)
    .rename_axis('invalid_value')
    .to_frame('count')
)
if len(invalid_loan_categories) > 0:
    invalid_loan_categories['percentage_%'] = (invalid_loan_categories['count'] / total_rows * 100).round(2)

print('\nInvalid LoanNr_ChkDgt categories:')
display(invalid_loan_categories)

# Duplicate values breakdown
duplicate_loan_values = (
    loan_std[duplicate_mask_loan]
    .value_counts(dropna=False)
    .rename_axis('LoanNr_ChkDgt_value')
    .to_frame('count')
)
if len(duplicate_loan_values) > 0:
    duplicate_loan_values['percentage_%'] = (duplicate_loan_values['count'] / total_rows * 100).round(2)

print('\nTop duplicated LoanNr_ChkDgt values:')
display(duplicate_loan_values.head(30))

print('\nSample duplicate LoanNr_ChkDgt rows (up to 20):')
display(df.loc[duplicate_mask_loan, ['LoanNr_ChkDgt']].head(20))

LoanNr_ChkDgt data quality summary:


,count,percentage_%
total_rows,20768,100.0
missing,0,0.0
invalid_not_10_digits,0,0.0
valid_10_digits,20768,100.0
duplicate_rows,0,0.0
duplicate_unique_values,0,0.0



LoanNr_ChkDgt value distribution (all categories):


LoanNr_ChkDgt data quality summary:


,count,percentage_%
total_rows,20768,100.0
missing,0,0.0
invalid_not_10_digits,0,0.0
valid_10_digits,20768,100.0
duplicate_rows,0,0.0
duplicate_unique_values,0,0.0


...

,count
invalid_value,



Top duplicated LoanNr_ChkDgt values:


,count
LoanNr_ChkDgt_value,



Sample duplicate LoanNr_ChkDgt rows (up to 20):


,LoanNr_ChkDgt


# Name
Text 	Name of the business owner or borrower. 	High-cardinality text; often not ideal for direct modeling.

In [3]:
# Name audit: counts, percentages, missing values, invalid categories, and duplicates
name_raw = df['Name']

# Standardize to string for consistent category checks
name_std = name_raw.astype('string').str.strip().str.upper()

# Missing includes true NaN and blank strings
missing_mask_name = name_raw.isna() | name_std.isna() | (name_std == '')

# Invalid Name categories: placeholder tokens or values with no alphabetic characters
invalid_values_name = {'N/A', 'NA', 'NONE', 'NULL', 'UNKNOWN', 'UNK', '?', '-'}
has_alpha_name = name_std.str.contains(r'[A-Z]', na=False)
valid_mask_name = (~missing_mask_name) & (~name_std.isin(invalid_values_name)) & has_alpha_name
invalid_mask_name = (~missing_mask_name) & (~valid_mask_name)

# Duplicate check over valid non-missing standardized values
duplicate_mask_name = valid_mask_name & name_std.duplicated(keep=False)

total_rows = len(df)
summary_name = pd.DataFrame(
    {'count': [
        total_rows,
        int(missing_mask_name.sum()),
        int(invalid_mask_name.sum()),
        int(valid_mask_name.sum()),
        int(duplicate_mask_name.sum()),
        int(name_std[duplicate_mask_name].nunique(dropna=True))
    ]},
    index=[
        'total_rows',
        'missing',
        'invalid_name',
        'valid_name',
        'duplicate_rows',
        'duplicate_unique_values'
    ]
)
summary_name['percentage_%'] = (summary_name['count'] / total_rows * 100).round(2)

print('Name data quality summary:')
display(summary_name)

# Full distribution (including missing shown as MISSING)
name_distribution = (
    name_std.fillna('MISSING')
    .replace({'': 'MISSING'})
    .value_counts(dropna=False)
    .rename_axis('Name_value')
    .to_frame('count')
)
name_distribution['percentage_%'] = (name_distribution['count'] / total_rows * 100).round(2)

print('\nName value distribution (all categories):')
display(name_distribution.head(30))

# Invalid category breakdown
invalid_name_categories = (
    name_std[invalid_mask_name]
    .value_counts(dropna=False)
    .rename_axis('invalid_Name_value')
    .to_frame('count')
)
if len(invalid_name_categories) > 0:
    invalid_name_categories['percentage_%'] = (
        invalid_name_categories['count'] / total_rows * 100
    ).round(2)

print('\nInvalid Name categories:')
display(invalid_name_categories)

# Duplicate values breakdown
duplicate_name_values = (
    name_std[duplicate_mask_name]
    .value_counts(dropna=False)
    .rename_axis('Name_value')
    .to_frame('count')
)
if len(duplicate_name_values) > 0:
    duplicate_name_values['percentage_%'] = (
        duplicate_name_values['count'] / total_rows * 100
    ).round(2)

print('\nTop duplicated Name values:')
display(duplicate_name_values.head(30))

print('\nSample invalid Name rows (up to 20):')
display(df.loc[invalid_mask_name, ['Name', 'LowDoc', 'Accept']].head(20))

print('\nSample duplicate Name rows (up to 20):')
display(df.loc[duplicate_mask_name, ['Name', 'LowDoc', 'Accept']].head(20))

Name data quality summary:


,count,percentage_%
total_rows,20768,100.00
missing,0,0.00
invalid_name,1,0.00
valid_name,20767,100.00
duplicate_rows,3107,14.96
duplicate_unique_values,1307,6.29



Name value distribution (all categories):


,count,percentage_%
Name_value,,
SUBWAY,43,0.21
QUIZNO'S SUBS,17,0.08
QUIZNO'S CLASSIC SUBS,17,0.08
DUNKIN' DONUTS,15,0.07
COLD STONE CREAMERY,14,0.07
QUIZNO'S,13,0.06
PEARLE VISION,12,0.06
DOMINO'S PIZZA,12,0.06
DUNKIN DONUTS,12,0.06



Invalid Name categories:


...

,count,percentage_%
Name_value,,
SUBWAY,43,0.21
QUIZNO'S SUBS,17,0.08
QUIZNO'S CLASSIC SUBS,17,0.08
DUNKIN' DONUTS,15,0.07
COLD STONE CREAMERY,14,0.07
QUIZNO'S,13,0.06
PEARLE VISION,12,0.06
DOMINO'S PIZZA,12,0.06
DUNKIN DONUTS,12,0.06



Sample invalid Name rows (up to 20):


,Name,LowDoc,Accept
15398,na,N,1



Sample duplicate Name rows (up to 20):


,Name,LowDoc,Accept
7,ALPHAGRAPHICS PRINTSHOPS,N,1
24,HAROLD'S CHICKEN SHACK,Y,0
36,ARMOR INSURANCE & FINANCIAL SE,N,0
37,"CONCRETE SUPPLY OF TOLONO, INC",N,1
38,"ROUND GROUND, INC.",N,1
43,PROGRESSIVE TECHNOLOGY SOLUTIO,N,1
44,Forest City Header Die & Machi,N,1
51,"PRAYOSHA-1, INC. DBA PIK KWIK",N,1
53,"INNOSYS COMMUNICATIONS, INC.",N,1
63,"SMG SECURITY SYSTEMS, INC.",N,1


# City
Text 	Borrower's city. 	Location feature.

In [4]:
# City audit: counts, percentages, missing values, invalid categories, and duplicates
city_raw = df['City']

# Standardize to string for consistent category checks
city_std = city_raw.astype('string').str.strip().str.upper()

# Missing includes true NaN and blank strings
missing_mask_city = city_raw.isna() | city_std.isna() | (city_std == '')

# Invalid City categories: placeholder tokens or values with no alphabetic characters
invalid_values_city = {'N/A', 'NA', 'NONE', 'NULL', 'UNKNOWN', 'UNK', '?', '-'}
has_alpha_city = city_std.str.contains(r'[A-Z]', na=False)
valid_mask_city = (~missing_mask_city) & (~city_std.isin(invalid_values_city)) & has_alpha_city
invalid_mask_city = (~missing_mask_city) & (~valid_mask_city)

# Duplicate check over valid non-missing standardized values
duplicate_mask_city = valid_mask_city & city_std.duplicated(keep=False)

total_rows = len(df)
summary_city = pd.DataFrame(
    {'count': [
        total_rows,
        int(missing_mask_city.sum()),
        int(invalid_mask_city.sum()),
        int(valid_mask_city.sum()),
        int(duplicate_mask_city.sum()),
        int(city_std[duplicate_mask_city].nunique(dropna=True))
    ]},
    index=[
        'total_rows',
        'missing',
        'invalid_city',
        'valid_city',
        'duplicate_rows',
        'duplicate_unique_values'
    ]
)
summary_city['percentage_%'] = (summary_city['count'] / total_rows * 100).round(2)

print('City data quality summary:')
display(summary_city)

# Full distribution (including missing shown as MISSING)
city_distribution = (
    city_std.fillna('MISSING')
    .replace({'': 'MISSING'})
    .value_counts(dropna=False)
    .rename_axis('City_value')
    .to_frame('count')
)
city_distribution['percentage_%'] = (city_distribution['count'] / total_rows * 100).round(2)

print('\nCity value distribution (all categories):')
display(city_distribution.head(30))

# Invalid category breakdown
invalid_city_categories = (
    city_std[invalid_mask_city]
    .value_counts(dropna=False)
    .rename_axis('invalid_City_value')
    .to_frame('count')
)
if len(invalid_city_categories) > 0:
    invalid_city_categories['percentage_%'] = (
        invalid_city_categories['count'] / total_rows * 100
    ).round(2)

print('\nInvalid City categories:')
display(invalid_city_categories)

# Duplicate values breakdown
duplicate_city_values = (
    city_std[duplicate_mask_city]
    .value_counts(dropna=False)
    .rename_axis('City_value')
    .to_frame('count')
)
if len(duplicate_city_values) > 0:
    duplicate_city_values['percentage_%'] = (
        duplicate_city_values['count'] / total_rows * 100
    ).round(2)

print('\nTop duplicated City values:')
display(duplicate_city_values.head(30))

print('\nSample invalid City rows (up to 20):')
display(df.loc[invalid_mask_city, ['City', 'State', 'Accept']].head(20))

print('\nSample duplicate City rows (up to 20):')
display(df.loc[duplicate_mask_city, ['City', 'State', 'Accept']].head(20))

City data quality summary:


,count,percentage_%
total_rows,20768,100.00
missing,1,0.00
invalid_city,0,0.00
valid_city,20767,100.00
duplicate_rows,20342,97.95
duplicate_unique_values,694,3.34



City value distribution (all categories):


,count,percentage_%
City_value,,
CHICAGO,4562,21.97
SPRINGFIELD,464,2.23
ROCKFORD,366,1.76
NAPERVILLE,349,1.68
SCHAUMBURG,281,1.35
AURORA,265,1.28
PEORIA,262,1.26
CHAMPAIGN,227,1.09
ELGIN,227,1.09



Invalid City categories:


...

,count,percentage_%
City_value,,
CHICAGO,4562,21.97
SPRINGFIELD,464,2.23
ROCKFORD,366,1.76
NAPERVILLE,349,1.68
SCHAUMBURG,281,1.35
AURORA,265,1.28
PEORIA,262,1.26
CHAMPAIGN,227,1.09
ELGIN,227,1.09



Sample invalid City rows (up to 20):


,City,State,Accept



Sample duplicate City rows (up to 20):


,City,State,Accept
0,HARVEY,IL,0
1,CHICAGO,IL,1
2,ROCHELLE,IL,1
3,Loves park,IL,1
4,LISLE,IL,0
5,PEORIA HEIGHTS,IL,0
6,SKOKIE,IL,1
7,BLOOMINGDALE,IL,1
8,CHICAGO,IL,1
9,GRAYSLAKE (CORPORATE AND RR NA,IL,1


# State
Text 	Borrower's state. 	Two-letter state code in many cases.

In [5]:
# State audit: counts, percentages, missing values, invalid categories, and duplicates
state_raw = df['State']

# Standardize to string for consistent category checks
state_std = state_raw.astype('string').str.strip().str.upper()

# Missing includes true NaN and blank strings
missing_mask_state = state_raw.isna() | state_std.isna() | (state_std == '')

# Valid state categories: US postal state/territory codes (2 letters)
valid_values_state = {
    'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA',
    'HI','ID','IL','IN','IA','KS','KY','LA','ME','MD',
    'MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ',
    'NM','NY','NC','ND','OH','OK','OR','PA','RI','SC',
    'SD','TN','TX','UT','VT','VA','WA','WV','WI','WY',
    'DC','PR','VI','GU','AS','MP'
}
valid_mask_state = state_std.isin(valid_values_state)
invalid_mask_state = (~missing_mask_state) & (~valid_mask_state)

# Duplicate check over valid non-missing standardized values
duplicate_mask_state = (~missing_mask_state) & valid_mask_state & state_std.duplicated(keep=False)

total_rows = len(df)
summary_state = pd.DataFrame(
    {'count': [
        total_rows,
        int(missing_mask_state.sum()),
        int(invalid_mask_state.sum()),
        int(valid_mask_state.sum()),
        int(duplicate_mask_state.sum()),
        int(state_std[duplicate_mask_state].nunique(dropna=True))
    ]},
    index=[
        'total_rows',
        'missing',
        'invalid_non_state_code',
        'valid_state_code',
        'duplicate_rows',
        'duplicate_unique_values'
    ]
)
summary_state['percentage_%'] = (summary_state['count'] / total_rows * 100).round(2)

print('State data quality summary:')
display(summary_state)

# Full distribution (including missing shown as MISSING)
state_distribution = (
    state_std.fillna('MISSING')
    .replace({'': 'MISSING'})
    .value_counts(dropna=False)
    .rename_axis('State_value')
    .to_frame('count')
)
state_distribution['percentage_%'] = (state_distribution['count'] / total_rows * 100).round(2)

print('\nState value distribution (all categories):')
display(state_distribution)

# Invalid category breakdown
invalid_state_categories = (
    state_std[invalid_mask_state]
    .value_counts(dropna=False)
    .rename_axis('invalid_State_value')
    .to_frame('count')
)
if len(invalid_state_categories) > 0:
    invalid_state_categories['percentage_%'] = (
        invalid_state_categories['count'] / total_rows * 100
    ).round(2)

print('\nInvalid State categories:')
display(invalid_state_categories)

# Duplicate values breakdown
duplicate_state_values = (
    state_std[duplicate_mask_state]
    .value_counts(dropna=False)
    .rename_axis('State_value')
    .to_frame('count')
)
if len(duplicate_state_values) > 0:
    duplicate_state_values['percentage_%'] = (
        duplicate_state_values['count'] / total_rows * 100
    ).round(2)

print('\nDuplicated State values:')
display(duplicate_state_values)

print('\nSample invalid State rows (up to 20):')
display(df.loc[invalid_mask_state, ['State', 'City', 'Accept']].head(20))

print('\nSample duplicate State rows (up to 20):')
display(df.loc[duplicate_mask_state, ['State', 'City', 'Accept']].head(20))

State data quality summary:


,count,percentage_%
total_rows,20768,100.0
missing,0,0.0
invalid_non_state_code,0,0.0
valid_state_code,20768,100.0
duplicate_rows,20768,100.0
duplicate_unique_values,1,0.0



State value distribution (all categories):


,count,percentage_%
State_value,,
IL,20768,100.0



Invalid State categories:


...

,count,percentage_%
State_value,,
IL,20768,100.0



Sample invalid State rows (up to 20):


,State,City,Accept



Sample duplicate State rows (up to 20):


,State,City,Accept
0,IL,HARVEY,0
1,IL,CHICAGO,1
2,IL,ROCHELLE,1
3,IL,Loves park,1
4,IL,LISLE,0
5,IL,PEORIA HEIGHTS,0
6,IL,SKOKIE,1
7,IL,BLOOMINGDALE,1
8,IL,CHICAGO,1
9,IL,GRAYSLAKE (CORPORATE AND RR NA,1


# Bank
Text 	Name of the bank that handled the loan. 	Can capture lender-specific behavior.

In [6]:
# Bank audit: counts, percentages, missing values, invalid categories, and duplicates
bank_raw = df['Bank']

# Standardize to string for consistent category checks
bank_std = bank_raw.astype('string').str.strip().str.upper()

# Missing includes true NaN and blank strings
missing_mask_bank = bank_raw.isna() | bank_std.isna() | (bank_std == '')

# Invalid Bank categories: placeholder tokens or values with no alphabetic characters
invalid_values_bank = {'N/A', 'NA', 'NONE', 'NULL', 'UNKNOWN', 'UNK', '?', '-'}
has_alpha_bank = bank_std.str.contains(r'[A-Z]', na=False)
valid_mask_bank = (~missing_mask_bank) & (~bank_std.isin(invalid_values_bank)) & has_alpha_bank
invalid_mask_bank = (~missing_mask_bank) & (~valid_mask_bank)

# Duplicate check over valid non-missing standardized values
duplicate_mask_bank = valid_mask_bank & bank_std.duplicated(keep=False)

total_rows = len(df)
summary_bank = pd.DataFrame(
    {'count': [
        total_rows,
        int(missing_mask_bank.sum()),
        int(invalid_mask_bank.sum()),
        int(valid_mask_bank.sum()),
        int(duplicate_mask_bank.sum()),
        int(bank_std[duplicate_mask_bank].nunique(dropna=True))
    ]},
    index=[
        'total_rows',
        'missing',
        'invalid_bank',
        'valid_bank',
        'duplicate_rows',
        'duplicate_unique_values'
    ]
)
summary_bank['percentage_%'] = (summary_bank['count'] / total_rows * 100).round(2)

print('Bank data quality summary:')
display(summary_bank)

# Full distribution (including missing shown as MISSING)
bank_distribution = (
    bank_std.fillna('MISSING')
    .replace({'': 'MISSING'})
    .value_counts(dropna=False)
    .rename_axis('Bank_value')
    .to_frame('count')
)
bank_distribution['percentage_%'] = (bank_distribution['count'] / total_rows * 100).round(2)

print('\nBank value distribution (all categories):')
display(bank_distribution.head(30))

# Invalid category breakdown
invalid_bank_categories = (
    bank_std[invalid_mask_bank]
    .value_counts(dropna=False)
    .rename_axis('invalid_Bank_value')
    .to_frame('count')
)
if len(invalid_bank_categories) > 0:
    invalid_bank_categories['percentage_%'] = (
        invalid_bank_categories['count'] / total_rows * 100
    ).round(2)

print('\nInvalid Bank categories:')
display(invalid_bank_categories)

# Duplicate values breakdown
duplicate_bank_values = (
    bank_std[duplicate_mask_bank]
    .value_counts(dropna=False)
    .rename_axis('Bank_value')
    .to_frame('count')
)
if len(duplicate_bank_values) > 0:
    duplicate_bank_values['percentage_%'] = (
        duplicate_bank_values['count'] / total_rows * 100
    ).round(2)

print('\nTop duplicated Bank values:')
display(duplicate_bank_values.head(30))

print('\nSample invalid Bank rows (up to 20):')
display(df.loc[invalid_mask_bank, ['Bank', 'BankState', 'Accept']].head(20))

print('\nSample duplicate Bank rows (up to 20):')
display(df.loc[duplicate_mask_bank, ['Bank', 'BankState', 'Accept']].head(20))

Bank data quality summary:


,count,percentage_%
total_rows,20768,100.00
missing,55,0.26
invalid_bank,0,0.00
valid_bank,20713,99.74
duplicate_rows,20559,98.99
duplicate_unique_values,342,1.65



Bank value distribution (all categories):


,count,percentage_%
Bank_value,,
JPMORGAN CHASE BANK NATL ASSOC,3027,14.58
"PNC BANK, NATIONAL ASSOCIATION",1352,6.51
BBCN BANK,1352,6.51
CITIZENS BANK NATL ASSOC,1326,6.38
U.S. BANK NATIONAL ASSOCIATION,1294,6.23
BMO HARRIS BK NATL ASSOC,714,3.44
SMALL BUS. GROWTH CORP,668,3.22
BANK OF AMERICA NATL ASSOC,628,3.02
"SOMERCOR 504, INC.",573,2.76



Invalid Bank categories:


...

,count,percentage_%
Bank_value,,
JPMORGAN CHASE BANK NATL ASSOC,3027,14.58
"PNC BANK, NATIONAL ASSOCIATION",1352,6.51
BBCN BANK,1352,6.51
CITIZENS BANK NATL ASSOC,1326,6.38
U.S. BANK NATIONAL ASSOCIATION,1294,6.23
BMO HARRIS BK NATL ASSOC,714,3.44
SMALL BUS. GROWTH CORP,668,3.22
BANK OF AMERICA NATL ASSOC,628,3.02
"SOMERCOR 504, INC.",573,2.76



Sample invalid Bank rows (up to 20):


,Bank,BankState,Accept



Sample duplicate Bank rows (up to 20):


,Bank,BankState,Accept
0,JPMORGAN CHASE BANK NATL ASSOC,IL,0
1,JPMORGAN CHASE BANK NATL ASSOC,IL,1
2,BMO HARRIS BK NATL ASSOC,IL,1
3,ALPINE BANK & TRUST CO.,IL,1
4,JPMORGAN CHASE BANK NATL ASSOC,IL,0
5,HERITAGE BK OF CENT. ILLINOIS,IL,0
6,CITIZENS BANK NATL ASSOC,RI,1
7,WELLS FARGO BANK NATL ASSOC,SD,1
8,"SOMERCOR 504, INC.",IL,1
9,JPMORGAN CHASE BANK NATL ASSOC,IL,1


# BankState
Text 	State where the bank is located. 	May differ from borrower state.

In [7]:
# BankState audit: counts, percentages, missing values, invalid categories, and duplicates
bankstate_raw = df['BankState']

# Standardize to string for consistent category checks
bankstate_std = bankstate_raw.astype('string').str.strip().str.upper()

# Missing includes true NaN and blank strings
missing_mask_bankstate = bankstate_raw.isna() | bankstate_std.isna() | (bankstate_std == '')

# Valid categories: US postal state/territory codes
valid_values_bankstate = {
    'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA',
    'HI','ID','IL','IN','IA','KS','KY','LA','ME','MD',
    'MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ',
    'NM','NY','NC','ND','OH','OK','OR','PA','RI','SC',
    'SD','TN','TX','UT','VT','VA','WA','WV','WI','WY',
    'DC','PR','VI','GU','AS','MP'
}
valid_mask_bankstate = bankstate_std.isin(valid_values_bankstate)
invalid_mask_bankstate = (~missing_mask_bankstate) & (~valid_mask_bankstate)

# Duplicate check over valid non-missing standardized values
duplicate_mask_bankstate = (
    (~missing_mask_bankstate)
    & valid_mask_bankstate
    & bankstate_std.duplicated(keep=False)
)

total_rows = len(df)
summary_bankstate = pd.DataFrame(
    {'count': [
        total_rows,
        int(missing_mask_bankstate.sum()),
        int(invalid_mask_bankstate.sum()),
        int(valid_mask_bankstate.sum()),
        int(duplicate_mask_bankstate.sum()),
        int(bankstate_std[duplicate_mask_bankstate].nunique(dropna=True))
    ]},
    index=[
        'total_rows',
        'missing',
        'invalid_non_state_code',
        'valid_state_code',
        'duplicate_rows',
        'duplicate_unique_values'
    ]
)
summary_bankstate['percentage_%'] = (summary_bankstate['count'] / total_rows * 100).round(2)

print('BankState data quality summary:')
display(summary_bankstate)

# Full distribution (including missing shown as MISSING)
bankstate_distribution = (
    bankstate_std.fillna('MISSING')
    .replace({'': 'MISSING'})
    .value_counts(dropna=False)
    .rename_axis('BankState_value')
    .to_frame('count')
)
bankstate_distribution['percentage_%'] = (
    bankstate_distribution['count'] / total_rows * 100
).round(2)

print('\nBankState value distribution (all categories):')
display(bankstate_distribution)

# Invalid category breakdown
invalid_bankstate_categories = (
    bankstate_std[invalid_mask_bankstate]
    .value_counts(dropna=False)
    .rename_axis('invalid_BankState_value')
    .to_frame('count')
)
if len(invalid_bankstate_categories) > 0:
    invalid_bankstate_categories['percentage_%'] = (
        invalid_bankstate_categories['count'] / total_rows * 100
    ).round(2)

print('\nInvalid BankState categories:')
display(invalid_bankstate_categories)

# Duplicate values breakdown
duplicate_bankstate_values = (
    bankstate_std[duplicate_mask_bankstate]
    .value_counts(dropna=False)
    .rename_axis('BankState_value')
    .to_frame('count')
)
if len(duplicate_bankstate_values) > 0:
    duplicate_bankstate_values['percentage_%'] = (
        duplicate_bankstate_values['count'] / total_rows * 100
    ).round(2)

print('\nDuplicated BankState values:')
display(duplicate_bankstate_values)

print('\nSample invalid BankState rows (up to 20):')
display(df.loc[invalid_mask_bankstate, ['BankState', 'Bank', 'Accept']].head(20))

print('\nSample duplicate BankState rows (up to 20):')
display(df.loc[duplicate_mask_bankstate, ['BankState', 'Bank', 'Accept']].head(20))

BankState data quality summary:


,count,percentage_%
total_rows,20768,100.00
missing,57,0.27
invalid_non_state_code,0,0.00
valid_state_code,20711,99.73
duplicate_rows,20703,99.69
duplicate_unique_values,34,0.16



BankState value distribution (all categories):


,count,percentage_%
BankState_value,,
IL,13002,62.61
RI,1334,6.42
CA,1324,6.38
OH,1105,5.32
DE,663,3.19
VA,567,2.73
NC,443,2.13
SD,429,2.07
NY,348,1.68



Invalid BankState categories:


...

,count,percentage_%
BankState_value,,
IL,13002,62.61
RI,1334,6.42
CA,1324,6.38
OH,1105,5.32
DE,663,3.19
VA,567,2.73
NC,443,2.13
SD,429,2.07
NY,348,1.68



Sample invalid BankState rows (up to 20):


,BankState,Bank,Accept



Sample duplicate BankState rows (up to 20):


,BankState,Bank,Accept
0,IL,JPMORGAN CHASE BANK NATL ASSOC,0
1,IL,JPMORGAN CHASE BANK NATL ASSOC,1
2,IL,BMO HARRIS BK NATL ASSOC,1
3,IL,ALPINE BANK & TRUST CO.,1
4,IL,JPMORGAN CHASE BANK NATL ASSOC,0
5,IL,HERITAGE BK OF CENT. ILLINOIS,0
6,RI,CITIZENS BANK NATL ASSOC,1
7,SD,WELLS FARGO BANK NATL ASSOC,1
8,IL,"SOMERCOR 504, INC.",1
9,IL,JPMORGAN CHASE BANK NATL ASSOC,1


# ApprovalDate
Date/Time 	Date when the SBA commitment was approved. 	Can be converted into year/month/quarter features.

In [8]:
# ApprovalDate audit: counts, percentages, missing values, invalid format, and duplicates
approval_raw = df['ApprovalDate']

# Standardize to string for consistent category checks
approval_std = approval_raw.astype('string').str.strip().str.upper()

# Missing includes true NaN and blank strings
missing_mask_approval = approval_raw.isna() | approval_std.isna() | (approval_std == '')

# Parse dates to identify invalid non-missing values
approval_parsed = pd.to_datetime(approval_std, errors='coerce')
valid_mask_approval = (~missing_mask_approval) & approval_parsed.notna()
invalid_mask_approval = (~missing_mask_approval) & approval_parsed.isna()

# Duplicate check over valid parsed dates
duplicate_mask_approval = valid_mask_approval & approval_parsed.duplicated(keep=False)

total_rows = len(df)
summary_approval = pd.DataFrame(
    {'count': [
        total_rows,
        int(missing_mask_approval.sum()),
        int(invalid_mask_approval.sum()),
        int(valid_mask_approval.sum()),
        int(duplicate_mask_approval.sum()),
        int(approval_parsed[duplicate_mask_approval].nunique(dropna=True))
    ]},
    index=[
        'total_rows',
        'missing',
        'invalid_date_format',
        'valid_date',
        'duplicate_rows',
        'duplicate_unique_values'
    ]
)
summary_approval['percentage_%'] = (summary_approval['count'] / total_rows * 100).round(2)

print('ApprovalDate data quality summary:')
display(summary_approval)

# Full distribution (including missing shown as MISSING)
approval_distribution = (
    approval_std.fillna('MISSING')
    .replace({'': 'MISSING'})
    .value_counts(dropna=False)
    .rename_axis('ApprovalDate_value')
    .to_frame('count')
)
approval_distribution['percentage_%'] = (approval_distribution['count'] / total_rows * 100).round(2)

print('\nApprovalDate value distribution (all categories):')
display(approval_distribution.head(30))

# Invalid category breakdown
invalid_approval_categories = (
    approval_std[invalid_mask_approval]
    .value_counts(dropna=False)
    .rename_axis('invalid_ApprovalDate_value')
    .to_frame('count')
)
if len(invalid_approval_categories) > 0:
    invalid_approval_categories['percentage_%'] = (
        invalid_approval_categories['count'] / total_rows * 100
    ).round(2)

print('\nInvalid ApprovalDate categories:')
display(invalid_approval_categories)

# Duplicate values breakdown (using normalized parsed date)
duplicate_approval_values = (
    approval_parsed[duplicate_mask_approval]
    .dt.strftime('%Y-%m-%d')
    .value_counts(dropna=False)
    .rename_axis('ApprovalDate_parsed_value')
    .to_frame('count')
)
if len(duplicate_approval_values) > 0:
    duplicate_approval_values['percentage_%'] = (
        duplicate_approval_values['count'] / total_rows * 100
    ).round(2)

print('\nTop duplicated ApprovalDate values:')
display(duplicate_approval_values.head(30))

print('\nSample invalid ApprovalDate rows (up to 20):')
display(df.loc[invalid_mask_approval, ['ApprovalDate', 'DisbursementDate', 'Accept']].head(20))

print('\nSample duplicate ApprovalDate rows (up to 20):')
display(df.loc[duplicate_mask_approval, ['ApprovalDate', 'DisbursementDate', 'Accept']].head(20))

C:\Users\nyliz\AppData\Local\Temp\ipykernel_21416\1059624205.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  approval_parsed = pd.to_datetime(approval_std, errors='coerce')


ApprovalDate data quality summary:


,count,percentage_%
total_rows,20768,100.00
missing,0,0.00
invalid_date_format,0,0.00
valid_date,20768,100.00
duplicate_rows,19190,92.40
duplicate_unique_values,3898,18.77



ApprovalDate value distribution (all categories):


,count,percentage_%
ApprovalDate_value,,
30-JUN-05,26,0.13
18-MAR-05,24,0.12
17-DEC-04,23,0.11
12-OCT-04,23,0.11
15-JUN-07,23,0.11
17-MAR-06,22,0.11
18-APR-05,21,0.1
30-JAN-04,21,0.1
30-SEP-03,21,0.1


...

,count,percentage_%
ApprovalDate_parsed_value,,
2005-06-30,26,0.13
2005-03-18,24,0.12
2004-12-17,23,0.11
2004-10-12,23,0.11
2007-06-15,23,0.11
2006-03-17,22,0.11
2005-04-18,21,0.10
2004-01-30,21,0.10
2003-09-30,21,0.10



Sample invalid ApprovalDate rows (up to 20):


,ApprovalDate,DisbursementDate,Accept



Sample duplicate ApprovalDate rows (up to 20):


,ApprovalDate,DisbursementDate,Accept
0,9-Aug-96,31-Mar-97,0
1,10-Dec-07,31-Dec-07,1
2,23-May-96,30-Sep-96,1
3,4-Nov-10,1-Mar-11,1
4,3-May-07,31-May-07,0
5,11-Mar-03,31-Oct-03,0
6,24-Oct-05,30-Nov-05,1
7,9-Feb-93,31-Jul-93,1
8,24-May-99,17-Jan-01,1
9,16-Mar-04,31-Mar-04,1


# ApprovalFY
Text 	Fiscal year when the loan was approved. 	Time and macroeconomic context feature.

In [9]:
# ApprovalFY audit: counts, percentages, missing values, invalid categories, and duplicates
approvalfy_raw = df['ApprovalFY']

# Standardize to string for consistent category checks
approvalfy_std = approvalfy_raw.astype('string').str.strip().str.upper()

# Missing includes true NaN and blank strings
missing_mask_approvalfy = approvalfy_raw.isna() | approvalfy_std.isna() | (approvalfy_std == '')

# Parse numeric fiscal year and validate plausible range
approvalfy_num = pd.to_numeric(approvalfy_std, errors='coerce')

# Keep year bounds broad enough for historical/future data quality checks
valid_year_min = 1900
valid_year_max = pd.Timestamp.today().year + 1
valid_year_range_mask = approvalfy_num.between(valid_year_min, valid_year_max, inclusive='both')
valid_integer_year_mask = approvalfy_num.notna() & (approvalfy_num % 1 == 0)

valid_mask_approvalfy = (~missing_mask_approvalfy) & valid_integer_year_mask & valid_year_range_mask
invalid_mask_approvalfy = (~missing_mask_approvalfy) & (~valid_mask_approvalfy)

# Duplicate check over valid non-missing fiscal years
duplicate_mask_approvalfy = valid_mask_approvalfy & approvalfy_num.duplicated(keep=False)

total_rows = len(df)
summary_approvalfy = pd.DataFrame(
    {'count': [
        total_rows,
        int(missing_mask_approvalfy.sum()),
        int(invalid_mask_approvalfy.sum()),
        int(valid_mask_approvalfy.sum()),
        int(duplicate_mask_approvalfy.sum()),
        int(approvalfy_num[duplicate_mask_approvalfy].nunique(dropna=True))
    ]},
    index=[
        'total_rows',
        'missing',
        'invalid_non_integer_or_out_of_range_year',
        'valid_year',
        'duplicate_rows',
        'duplicate_unique_values'
    ]
)
summary_approvalfy['percentage_%'] = (summary_approvalfy['count'] / total_rows * 100).round(2)

print('ApprovalFY data quality summary:')
display(summary_approvalfy)

# Full distribution (including missing shown as MISSING)
approvalfy_distribution = (
    approvalfy_std.fillna('MISSING')
    .replace({'': 'MISSING'})
    .value_counts(dropna=False)
    .rename_axis('ApprovalFY_value')
    .to_frame('count')
)
approvalfy_distribution['percentage_%'] = (
    approvalfy_distribution['count'] / total_rows * 100
).round(2)

print('\nApprovalFY value distribution (all categories):')
display(approvalfy_distribution.head(30))

# Invalid category breakdown
invalid_approvalfy_categories = (
    approvalfy_std[invalid_mask_approvalfy]
    .value_counts(dropna=False)
    .rename_axis('invalid_ApprovalFY_value')
    .to_frame('count')
)
if len(invalid_approvalfy_categories) > 0:
    invalid_approvalfy_categories['percentage_%'] = (
        invalid_approvalfy_categories['count'] / total_rows * 100
    ).round(2)

print('\nInvalid ApprovalFY categories:')
display(invalid_approvalfy_categories)

# Duplicate values breakdown (normalized as integer year labels)
duplicate_approvalfy_values = (
    approvalfy_num[duplicate_mask_approvalfy]
    .astype('Int64')
    .astype('string')
    .value_counts(dropna=False)
    .rename_axis('ApprovalFY_value')
    .to_frame('count')
)
if len(duplicate_approvalfy_values) > 0:
    duplicate_approvalfy_values['percentage_%'] = (
        duplicate_approvalfy_values['count'] / total_rows * 100
    ).round(2)

print('\nDuplicated ApprovalFY values:')
display(duplicate_approvalfy_values)

print('\nSample invalid ApprovalFY rows (up to 20):')
display(df.loc[invalid_mask_approvalfy, ['ApprovalFY', 'ApprovalDate', 'Accept']].head(20))

print('\nSample duplicate ApprovalFY rows (up to 20):')
display(df.loc[duplicate_mask_approvalfy, ['ApprovalFY', 'ApprovalDate', 'Accept']].head(20))

ApprovalFY data quality summary:


,count,percentage_%
total_rows,20768,100.00
missing,0,0.00
invalid_non_integer_or_out_of_range_year,1,0.00
valid_year,20767,100.00
duplicate_rows,20766,99.99
duplicate_unique_values,43,0.21



ApprovalFY value distribution (all categories):


,count,percentage_%
ApprovalFY_value,,
2007,2312,11.13
2006,2284,11.0
2005,2106,10.14
2004,1282,6.17
2008,1226,5.9
1995,1193,5.74
2003,1028,4.95
1996,1001,4.82
1997,802,3.86



Invalid ApprovalFY categories:


...

,count,percentage_%
ApprovalFY_value,,
2007,2312,11.13
2006,2284,11.0
2005,2106,10.14
2004,1282,6.17
2008,1226,5.9
1995,1193,5.74
2003,1028,4.95
1996,1001,4.82
1997,802,3.86



Sample invalid ApprovalFY rows (up to 20):


,ApprovalFY,ApprovalDate,Accept
20389,1976A,29-Sep-76,0



Sample duplicate ApprovalFY rows (up to 20):


,ApprovalFY,ApprovalDate,Accept
0,1996,9-Aug-96,0
1,2008,10-Dec-07,1
2,1996,23-May-96,1
3,2011,4-Nov-10,1
4,2007,3-May-07,0
5,2003,11-Mar-03,0
6,2006,24-Oct-05,1
7,1993,9-Feb-93,1
8,1999,24-May-99,1
9,2004,16-Mar-04,1


# NoEmp
Number 	Number of employees in the business. 	Proxy for business size.

In [14]:
# NoEmp audit: counts, percentages, missing values, and invalid categories (empty, 0, negative)
noemp_raw = df['NoEmp']

# Standardize to string for consistent category checks
noemp_std = noemp_raw.astype('string').str.strip()

# Empty strings (separate category requested)
empty_mask_noemp = noemp_std.fillna('').eq('')

# Missing includes true NaN and blank strings
missing_mask_noemp = noemp_raw.isna() | empty_mask_noemp

# Numeric conversion for validity checks
noemp_num = pd.to_numeric(noemp_std, errors='coerce')

# Invalid categories requested by user
zero_mask_noemp = (~missing_mask_noemp) & noemp_num.eq(0)
negative_mask_noemp = (~missing_mask_noemp) & noemp_num.lt(0)
invalid_mask_noemp = empty_mask_noemp | zero_mask_noemp | negative_mask_noemp

# Valid NoEmp: non-missing and strictly positive numeric values
valid_mask_noemp = (~missing_mask_noemp) & noemp_num.gt(0)

total_rows = len(df)
summary_noemp = pd.DataFrame(
    {'count': [
        total_rows,
        int(missing_mask_noemp.sum()),
        int(empty_mask_noemp.sum()),
        int(zero_mask_noemp.sum()),
        int(negative_mask_noemp.sum()),
        int(invalid_mask_noemp.sum()),
        int(valid_mask_noemp.sum())
    ]},
    index=[
        'total_rows',
        'missing',
        'invalid_empty',
        'invalid_zero',
        'invalid_negative',
        'invalid_total',
        'valid_positive'
    ]
)
summary_noemp['percentage_%'] = (summary_noemp['count'] / total_rows * 100).round(2)

print('NoEmp data quality summary:')
display(summary_noemp)

# Full distribution (including missing shown as MISSING)
noemp_distribution = (
    noemp_std.fillna('MISSING')
    .replace({'': 'MISSING'})
    .value_counts(dropna=False)
    .rename_axis('NoEmp_value')
    .to_frame('count')
)
noemp_distribution['percentage_%'] = (noemp_distribution['count'] / total_rows * 100).round(2)

print('\nNoEmp value distribution (all categories):')
display(noemp_distribution.head(30))

# Explicit invalid category table requested (empty, 0, negative)
invalid_noemp_categories = pd.DataFrame(
    {'count': [
        int(empty_mask_noemp.sum()),
        int(zero_mask_noemp.sum()),
        int(negative_mask_noemp.sum())
    ]},
    index=['empty', '0', 'negative']
)
invalid_noemp_categories['percentage_%'] = (
    invalid_noemp_categories['count'] / total_rows * 100
).round(2)

print('\nInvalid NoEmp categories (empty, 0, negative):')
display(invalid_noemp_categories)

print('\nSample invalid NoEmp rows (up to 20):')
display(df.loc[invalid_mask_noemp, ['NoEmp', 'Name', 'Accept']].head(20))

NoEmp data quality summary:


,count,percentage_%
total_rows,20768,100.00
missing,0,0.00
invalid_empty,0,0.00
invalid_zero,212,1.02
invalid_negative,0,0.00
invalid_total,212,1.02
valid_positive,20556,98.98



NoEmp value distribution (all categories):


,count,percentage_%
NoEmp_value,,
1,3460,16.66
2,3320,15.99
3,2206,10.62
4,1840,8.86
5,1462,7.04
6,1052,5.07
10,753,3.63
8,710,3.42
7,672,3.24



Invalid NoEmp categories (empty, 0, negative):


,count,percentage_%
empty,0,0.00
0,212,1.02
negative,0,0.00



Sample invalid NoEmp rows (up to 20):


,NoEmp,Name,Accept
70,0,"Jackson Avenue Subs, Inc. Iqba",1
160,0,"InterArts Management, Inc.",1
239,0,"Eagle Polymers, Inc.",1
260,0,"Murchison & Phillips, Inc.",0
358,0,JLZ Consulting LLC,1
394,0,"Tamuck, Inc.",1
414,0,Law Office of Guadalupe Sandov,1
586,0,"Kraemerica Enterprises, LLC db",1
740,0,"Solar Max Tan, Inc.",0
796,0,"MCS Leasing, Ltd.",1


# NewExist
Text 	Whether the business is new or existing. 	1 = Existing business, 2 = New business.

In [15]:
# NewExist audit: counts, percentages, missing values, and invalid categories
# Valid values are only 1 (existing) and 2 (new)
newexist_raw = df['NewExist']

# Standardize to string for consistent category checks
newexist_std = newexist_raw.astype('string').str.strip().str.upper()

# Empty strings (explicit invalid category requested)
empty_mask_newexist = newexist_std.fillna('').eq('')

# Missing includes true NaN and blank strings
missing_mask_newexist = newexist_raw.isna() | empty_mask_newexist

# Numeric conversion for numeric validity checks
newexist_num = pd.to_numeric(newexist_std, errors='coerce')

# Invalid categories requested
zero_mask_newexist = (~missing_mask_newexist) & newexist_num.eq(0)
negative_mask_newexist = (~missing_mask_newexist) & newexist_num.lt(0)
greater2_mask_newexist = (~missing_mask_newexist) & newexist_num.gt(2)
letters_mask_newexist = (~missing_mask_newexist) & newexist_std.str.contains(r'[A-Z]', na=False)

# Valid mask: strictly 1 or 2
valid_values_newexist = {1, 2}
valid_mask_newexist = (~missing_mask_newexist) & newexist_num.isin(valid_values_newexist)

# Everything non-missing and not valid is invalid
invalid_mask_newexist = (~missing_mask_newexist) & (~valid_mask_newexist)

# Optional catch-all for invalid values not covered by the requested categories
other_invalid_mask_newexist = invalid_mask_newexist & (
    ~zero_mask_newexist
    & ~negative_mask_newexist
    & ~greater2_mask_newexist
    & ~letters_mask_newexist
)

total_rows = len(df)
summary_newexist = pd.DataFrame(
    {'count': [
        total_rows,
        int(missing_mask_newexist.sum()),
        int(empty_mask_newexist.sum()),
        int(zero_mask_newexist.sum()),
        int(negative_mask_newexist.sum()),
        int(greater2_mask_newexist.sum()),
        int(letters_mask_newexist.sum()),
        int(other_invalid_mask_newexist.sum()),
        int(invalid_mask_newexist.sum()),
        int(valid_mask_newexist.sum())
    ]},
    index=[
        'total_rows',
        'missing',
        'invalid_empty',
        'invalid_0',
        'invalid_negative',
        'invalid_greater_than_2',
        'invalid_letters',
        'invalid_other_non_12',
        'invalid_total',
        'valid_1_or_2'
    ]
)
summary_newexist['percentage_%'] = (summary_newexist['count'] / total_rows * 100).round(2)

print('NewExist data quality summary:')
display(summary_newexist)

# Full distribution (including missing shown as MISSING)
newexist_distribution = (
    newexist_std.fillna('MISSING')
    .replace({'': 'MISSING'})
    .value_counts(dropna=False)
    .rename_axis('NewExist_value')
    .to_frame('count')
)
newexist_distribution['percentage_%'] = (newexist_distribution['count'] / total_rows * 100).round(2)

print('\nNewExist value distribution (all categories):')
display(newexist_distribution)

# Explicit invalid-category table requested
invalid_newexist_categories = pd.DataFrame(
    {'count': [
        int(empty_mask_newexist.sum()),
        int(zero_mask_newexist.sum()),
        int(negative_mask_newexist.sum()),
        int(greater2_mask_newexist.sum()),
        int(letters_mask_newexist.sum())
    ]},
    index=['empty', '0', 'negative', '>2', 'letters']
)
invalid_newexist_categories['percentage_%'] = (
    invalid_newexist_categories['count'] / total_rows * 100
).round(2)

print('\nInvalid NewExist categories (empty, 0, negative, >2, letters):')
display(invalid_newexist_categories)

print('\nSample invalid NewExist rows (up to 20):')
display(df.loc[invalid_mask_newexist, ['NewExist', 'NoEmp', 'Accept']].head(20))

NewExist data quality summary:


,count,percentage_%
total_rows,20768,100.00
missing,3,0.01
invalid_empty,3,0.01
invalid_0,10,0.05
invalid_negative,0,0.00
invalid_greater_than_2,0,0.00
invalid_letters,0,0.00
invalid_other_non_12,0,0.00
invalid_total,10,0.05
valid_1_or_2,20755,99.94



NewExist value distribution (all categories):


,count,percentage_%
NewExist_value,,
1.0,14178,68.27
2.0,6577,31.67
0.0,10,0.05
MISSING,3,0.01



Invalid NewExist categories (empty, 0, negative, >2, letters):


,count,percentage_%
empty,3,0.01
0,10,0.05
negative,0,0.00
>2,0,0.00
letters,0,0.00



Sample invalid NewExist rows (up to 20):


,NewExist,NoEmp,Accept
1080,0.0,8,1
2995,0.0,2,0
3226,0.0,2,1
3635,0.0,10,1
4503,0.0,2,1
6016,0.0,3,1
6630,0.0,2,1
10090,0.0,4,1
11339,0.0,3,1
12666,0.0,1,1


# CreateJob
Number 	Number of jobs expected to be created by the loan. 	Program impact indicator.

In [16]:
# CreateJob audit: counts, percentages, missing values, and invalid categories (empty, 0, negative, letters)
createjob_raw = df['CreateJob']

# Standardize to string for consistent category checks
createjob_std = createjob_raw.astype('string').str.strip().str.upper()

# Empty strings (explicit invalid category requested)
empty_mask_createjob = createjob_std.fillna('').eq('')

# Missing includes true NaN and blank strings
missing_mask_createjob = createjob_raw.isna() | empty_mask_createjob

# Numeric conversion for numeric validity checks
createjob_num = pd.to_numeric(createjob_std, errors='coerce')

# Invalid categories requested
zero_mask_createjob = (~missing_mask_createjob) & createjob_num.eq(0)
negative_mask_createjob = (~missing_mask_createjob) & createjob_num.lt(0)
letters_mask_createjob = (~missing_mask_createjob) & createjob_std.str.contains(r'[A-Z]', na=False)

invalid_mask_createjob = (
    empty_mask_createjob
    | zero_mask_createjob
    | negative_mask_createjob
    | letters_mask_createjob
)

# Valid CreateJob: non-missing and strictly positive numeric values
valid_mask_createjob = (~missing_mask_createjob) & createjob_num.gt(0)

total_rows = len(df)
summary_createjob = pd.DataFrame(
    {'count': [
        total_rows,
        int(missing_mask_createjob.sum()),
        int(empty_mask_createjob.sum()),
        int(zero_mask_createjob.sum()),
        int(negative_mask_createjob.sum()),
        int(letters_mask_createjob.sum()),
        int(invalid_mask_createjob.sum()),
        int(valid_mask_createjob.sum())
    ]},
    index=[
        'total_rows',
        'missing',
        'invalid_empty',
        'invalid_0',
        'invalid_negative',
        'invalid_letters',
        'invalid_total',
        'valid_positive'
    ]
)
summary_createjob['percentage_%'] = (summary_createjob['count'] / total_rows * 100).round(2)

print('CreateJob data quality summary:')
display(summary_createjob)

# Full distribution (including missing shown as MISSING)
createjob_distribution = (
    createjob_std.fillna('MISSING')
    .replace({'': 'MISSING'})
    .value_counts(dropna=False)
    .rename_axis('CreateJob_value')
    .to_frame('count')
)
createjob_distribution['percentage_%'] = (createjob_distribution['count'] / total_rows * 100).round(2)

print('\nCreateJob value distribution (all categories):')
display(createjob_distribution.head(30))

# Explicit invalid category table requested
invalid_createjob_categories = pd.DataFrame(
    {'count': [
        int(empty_mask_createjob.sum()),
        int(zero_mask_createjob.sum()),
        int(negative_mask_createjob.sum()),
        int(letters_mask_createjob.sum())
    ]},
    index=['empty', '0', 'negative', 'letters']
)
invalid_createjob_categories['percentage_%'] = (
    invalid_createjob_categories['count'] / total_rows * 100
).round(2)

print('\nInvalid CreateJob categories (empty, 0, negative, letters):')
display(invalid_createjob_categories)

print('\nSample invalid CreateJob rows (up to 20):')
display(df.loc[invalid_mask_createjob, ['CreateJob', 'NoEmp', 'Accept']].head(20))

CreateJob data quality summary:


,count,percentage_%
total_rows,20768,100.00
missing,0,0.00
invalid_empty,0,0.00
invalid_0,13919,67.02
invalid_negative,0,0.00
invalid_letters,0,0.00
invalid_total,13919,67.02
valid_positive,6849,32.98



CreateJob value distribution (all categories):


,count,percentage_%
CreateJob_value,,
0,13919,67.02
1,1773,8.54
2,1475,7.1
3,669,3.22
4,527,2.54
5,444,2.14
10,330,1.59
6,259,1.25
8,165,0.79



Invalid CreateJob categories (empty, 0, negative, letters):


,count,percentage_%
empty,0,0.00
0,13919,67.02
negative,0,0.00
letters,0,0.00



Sample invalid CreateJob rows (up to 20):


,CreateJob,NoEmp,Accept
0,0,28,0
2,0,6,1
3,0,5,1
5,0,11,0
7,0,5,1
12,0,3,1
16,0,1,1
18,0,113,1
19,0,1,1
20,0,3,1


# RetainedJob
Number 	Number of jobs expected to be retained. 	Program impact indicator.

In [17]:
# RetainedJob audit: counts, percentages, missing values, and invalid categories (empty, 0, negative, letters)
retainedjob_raw = df['RetainedJob']

# Standardize to string for consistent category checks
retainedjob_std = retainedjob_raw.astype('string').str.strip().str.upper()

# Empty strings (explicit invalid category requested)
empty_mask_retainedjob = retainedjob_std.fillna('').eq('')

# Missing includes true NaN and blank strings
missing_mask_retainedjob = retainedjob_raw.isna() | empty_mask_retainedjob

# Numeric conversion for numeric validity checks
retainedjob_num = pd.to_numeric(retainedjob_std, errors='coerce')

# Invalid categories requested
zero_mask_retainedjob = (~missing_mask_retainedjob) & retainedjob_num.eq(0)
negative_mask_retainedjob = (~missing_mask_retainedjob) & retainedjob_num.lt(0)
letters_mask_retainedjob = (~missing_mask_retainedjob) & retainedjob_std.str.contains(r'[A-Z]', na=False)

invalid_mask_retainedjob = (
    empty_mask_retainedjob
    | zero_mask_retainedjob
    | negative_mask_retainedjob
    | letters_mask_retainedjob
)

# Valid RetainedJob: non-missing and strictly positive numeric values
valid_mask_retainedjob = (~missing_mask_retainedjob) & retainedjob_num.gt(0)

total_rows = len(df)
summary_retainedjob = pd.DataFrame(
    {'count': [
        total_rows,
        int(missing_mask_retainedjob.sum()),
        int(empty_mask_retainedjob.sum()),
        int(zero_mask_retainedjob.sum()),
        int(negative_mask_retainedjob.sum()),
        int(letters_mask_retainedjob.sum()),
        int(invalid_mask_retainedjob.sum()),
        int(valid_mask_retainedjob.sum())
    ]},
    index=[
        'total_rows',
        'missing',
        'invalid_empty',
        'invalid_0',
        'invalid_negative',
        'invalid_letters',
        'invalid_total',
        'valid_positive'
    ]
)
summary_retainedjob['percentage_%'] = (summary_retainedjob['count'] / total_rows * 100).round(2)

print('RetainedJob data quality summary:')
display(summary_retainedjob)

# Full distribution (including missing shown as MISSING)
retainedjob_distribution = (
    retainedjob_std.fillna('MISSING')
    .replace({'': 'MISSING'})
    .value_counts(dropna=False)
    .rename_axis('RetainedJob_value')
    .to_frame('count')
)
retainedjob_distribution['percentage_%'] = (
    retainedjob_distribution['count'] / total_rows * 100
).round(2)

print('\nRetainedJob value distribution (all categories):')
display(retainedjob_distribution.head(30))

# Explicit invalid category table requested
invalid_retainedjob_categories = pd.DataFrame(
    {'count': [
        int(empty_mask_retainedjob.sum()),
        int(zero_mask_retainedjob.sum()),
        int(negative_mask_retainedjob.sum()),
        int(letters_mask_retainedjob.sum())
    ]},
    index=['empty', '0', 'negative', 'letters']
)
invalid_retainedjob_categories['percentage_%'] = (
    invalid_retainedjob_categories['count'] / total_rows * 100
).round(2)

print('\nInvalid RetainedJob categories (empty, 0, negative, letters):')
display(invalid_retainedjob_categories)

print('\nSample invalid RetainedJob rows (up to 20):')
display(df.loc[invalid_mask_retainedjob, ['RetainedJob', 'CreateJob', 'Accept']].head(20))

RetainedJob data quality summary:


,count,percentage_%
total_rows,20768,100.00
missing,0,0.00
invalid_empty,0,0.00
invalid_0,10484,50.48
invalid_negative,0,0.00
invalid_letters,0,0.00
invalid_total,10484,50.48
valid_positive,10284,49.52



RetainedJob value distribution (all categories):


,count,percentage_%
RetainedJob_value,,
0,10484,50.48
1,2119,10.2
2,1915,9.22
3,1238,5.96
4,920,4.43
5,785,3.78
6,536,2.58
10,347,1.67
7,337,1.62



Invalid RetainedJob categories (empty, 0, negative, letters):


,count,percentage_%
empty,0,0.00
0,10484,50.48
negative,0,0.00
letters,0,0.00



Sample invalid RetainedJob rows (up to 20):


,RetainedJob,CreateJob,Accept
0,0,0,0
2,0,0,1
5,0,0,0
7,0,0,1
8,0,6,1
10,0,6,0
11,0,3,1
12,0,0,1
14,0,22,1
18,0,0,1


# FranchiseCode
Text 	Code indicating franchise relationship. 	00000 or 00001 often means no franchise.

In [18]:
# FranchiseCode audit: counts, percentages, missing values, invalid categories, and duplicates
franchise_raw = df['FranchiseCode']

# Standardize to string for consistent category checks
franchise_std = franchise_raw.astype('string').str.strip().str.upper()

# Empty strings (explicit invalid category requested)
empty_mask_franchise = franchise_std.fillna('').eq('')

# Missing includes true NaN and blank strings
missing_mask_franchise = franchise_raw.isna() | empty_mask_franchise

# Numeric conversion for numeric validity checks
franchise_num = pd.to_numeric(franchise_std, errors='coerce')

# Invalid categories requested
zero_mask_franchise = (~missing_mask_franchise) & franchise_num.eq(0)
negative_mask_franchise = (~missing_mask_franchise) & franchise_num.lt(0)
letters_mask_franchise = (~missing_mask_franchise) & franchise_std.str.contains(r'[A-Z]', na=False)

invalid_mask_franchise = (
    empty_mask_franchise
    | zero_mask_franchise
    | negative_mask_franchise
    | letters_mask_franchise
)

# Optional catch-all for non-missing values that are not numeric and not letter-based
other_invalid_mask_franchise = (
    (~missing_mask_franchise)
    & franchise_num.isna()
    & (~letters_mask_franchise)
)

# Valid FranchiseCode: non-missing, numeric, and strictly positive
valid_mask_franchise = (~missing_mask_franchise) & franchise_num.gt(0)

# Duplicate check over valid non-missing standardized values
duplicate_mask_franchise = valid_mask_franchise & franchise_std.duplicated(keep=False)

total_rows = len(df)
summary_franchise = pd.DataFrame(
    {'count': [
        total_rows,
        int(missing_mask_franchise.sum()),
        int(empty_mask_franchise.sum()),
        int(zero_mask_franchise.sum()),
        int(negative_mask_franchise.sum()),
        int(letters_mask_franchise.sum()),
        int(other_invalid_mask_franchise.sum()),
        int(invalid_mask_franchise.sum()),
        int(valid_mask_franchise.sum()),
        int(duplicate_mask_franchise.sum()),
        int(franchise_std[duplicate_mask_franchise].nunique(dropna=True))
    ]},
    index=[
        'total_rows',
        'missing',
        'invalid_empty',
        'invalid_0',
        'invalid_negative',
        'invalid_letters',
        'invalid_other_non_numeric',
        'invalid_total_requested',
        'valid_positive_numeric',
        'duplicate_rows',
        'duplicate_unique_values'
    ]
)
summary_franchise['percentage_%'] = (summary_franchise['count'] / total_rows * 100).round(2)

print('FranchiseCode data quality summary:')
display(summary_franchise)

# Full distribution (including missing shown as MISSING)
franchise_distribution = (
    franchise_std.fillna('MISSING')
    .replace({'': 'MISSING'})
    .value_counts(dropna=False)
    .rename_axis('FranchiseCode_value')
    .to_frame('count')
)
franchise_distribution['percentage_%'] = (
    franchise_distribution['count'] / total_rows * 100
).round(2)

print('\nFranchiseCode value distribution (all categories):')
display(franchise_distribution.head(30))

# Explicit invalid category table requested
invalid_franchise_categories = pd.DataFrame(
    {'count': [
        int(empty_mask_franchise.sum()),
        int(zero_mask_franchise.sum()),
        int(negative_mask_franchise.sum()),
        int(letters_mask_franchise.sum())
    ]},
    index=['empty', '0', 'negative', 'letters']
)
invalid_franchise_categories['percentage_%'] = (
    invalid_franchise_categories['count'] / total_rows * 100
).round(2)

print('\nInvalid FranchiseCode categories (empty, 0, negative, letters):')
display(invalid_franchise_categories)

# Duplicate values breakdown
duplicate_franchise_values = (
    franchise_std[duplicate_mask_franchise]
    .value_counts(dropna=False)
    .rename_axis('FranchiseCode_value')
    .to_frame('count')
)
if len(duplicate_franchise_values) > 0:
    duplicate_franchise_values['percentage_%'] = (
        duplicate_franchise_values['count'] / total_rows * 100
    ).round(2)

print('\nTop duplicated FranchiseCode values:')
display(duplicate_franchise_values.head(30))

print('\nSample invalid FranchiseCode rows (up to 20):')
display(df.loc[invalid_mask_franchise | other_invalid_mask_franchise, ['FranchiseCode', 'NewExist', 'Accept']].head(20))

print('\nSample duplicate FranchiseCode rows (up to 20):')
display(df.loc[duplicate_mask_franchise, ['FranchiseCode', 'NewExist', 'Accept']].head(20))

FranchiseCode data quality summary:


,count,percentage_%
total_rows,20768,100.00
missing,0,0.00
invalid_empty,0,0.00
invalid_0,6514,31.37
invalid_negative,0,0.00
invalid_letters,0,0.00
invalid_other_non_numeric,0,0.00
invalid_total_requested,6514,31.37
valid_positive_numeric,14254,68.63
duplicate_rows,14038,67.59



FranchiseCode value distribution (all categories):


,count,percentage_%
FranchiseCode_value,,
1,12899,62.11
0,6514,31.37
78760,113,0.54
68020,56,0.27
25650,44,0.21
51702,34,0.16
21780,28,0.13
43579,20,0.1
21425,19,0.09



Invalid FranchiseCode categories (empty, 0, negative, letters):


...

,count,percentage_%
FranchiseCode_value,,
1,12899,62.11
78760,113,0.54
68020,56,0.27
25650,44,0.21
51702,34,0.16
21780,28,0.13
43579,20,0.1
21425,19,0.09
52875,18,0.09



Sample invalid FranchiseCode rows (up to 20):


,FranchiseCode,NewExist,Accept
1,0,2.0,1
3,0,2.0,1
4,0,1.0,0
6,0,1.0,1
15,0,1.0,0
18,0,1.0,1
21,0,1.0,0
33,0,1.0,1
34,0,1.0,0
35,0,1.0,1



Sample duplicate FranchiseCode rows (up to 20):


,FranchiseCode,NewExist,Accept
0,1,1.0,0
2,1,2.0,1
5,10481,2.0,0
7,3512,1.0,1
8,1,1.0,1
9,1,1.0,1
10,1,2.0,0
12,1,2.0,1
13,1,1.0,1
14,1,1.0,1


# UrbanRural
Text 	Area type where the business operates. 	1 = Urban, 2 = Rural, 0 = Undefined.

In [19]:
# UrbanRural audit: counts, percentages, missing values, invalid categories, and duplicates
# Accepted values: 1 (Urban), 2 (Rural), 0 (Undefined)
urban_raw = df['UrbanRural']

# Standardize to string for consistent category checks
urban_std = urban_raw.astype('string').str.strip().str.upper()

# Empty strings (explicit invalid category requested)
empty_mask_urban = urban_std.fillna('').eq('')

# Missing includes true NaN and blank strings
missing_mask_urban = urban_raw.isna() | empty_mask_urban

# Numeric conversion for numeric validity checks
urban_num = pd.to_numeric(urban_std, errors='coerce')

# Invalid categories requested
zero_mask_urban = (~missing_mask_urban) & urban_num.eq(0)
negative_mask_urban = (~missing_mask_urban) & urban_num.lt(0)
letters_mask_urban = (~missing_mask_urban) & urban_std.str.contains(r'[A-Z]', na=False)

# Valid values are 0, 1, 2
valid_values_urban = {0, 1, 2}
valid_mask_urban = (~missing_mask_urban) & urban_num.isin(valid_values_urban)
invalid_mask_urban = (~missing_mask_urban) & (~valid_mask_urban)

# Optional catch-all for invalid non-missing values not covered by requested categories
other_invalid_mask_urban = invalid_mask_urban & (
    ~zero_mask_urban
    & ~negative_mask_urban
    & ~letters_mask_urban
)

# Duplicate check over valid non-missing standardized values
duplicate_mask_urban = valid_mask_urban & urban_std.duplicated(keep=False)

total_rows = len(df)
summary_urban = pd.DataFrame(
    {'count': [
        total_rows,
        int(missing_mask_urban.sum()),
        int(empty_mask_urban.sum()),
        int(zero_mask_urban.sum()),
        int(negative_mask_urban.sum()),
        int(letters_mask_urban.sum()),
        int(other_invalid_mask_urban.sum()),
        int(invalid_mask_urban.sum()),
        int(valid_mask_urban.sum()),
        int(duplicate_mask_urban.sum()),
        int(urban_std[duplicate_mask_urban].nunique(dropna=True))
    ]},
    index=[
        'total_rows',
        'missing',
        'invalid_empty',
        'invalid_0',
        'invalid_negative',
        'invalid_letters',
        'invalid_other_non_012',
        'invalid_total',
        'valid_0_1_2',
        'duplicate_rows',
        'duplicate_unique_values'
    ]
)
summary_urban['percentage_%'] = (summary_urban['count'] / total_rows * 100).round(2)

print('UrbanRural data quality summary:')
display(summary_urban)

# Full distribution (including missing shown as MISSING)
urban_distribution = (
    urban_std.fillna('MISSING')
    .replace({'': 'MISSING'})
    .value_counts(dropna=False)
    .rename_axis('UrbanRural_value')
    .to_frame('count')
)
urban_distribution['percentage_%'] = (urban_distribution['count'] / total_rows * 100).round(2)

print('\nUrbanRural value distribution (all categories):')
display(urban_distribution)

# Explicit invalid category table requested
invalid_urban_categories = pd.DataFrame(
    {'count': [
        int(empty_mask_urban.sum()),
        int(zero_mask_urban.sum()),
        int(negative_mask_urban.sum()),
        int(letters_mask_urban.sum())
    ]},
    index=['empty', '0', 'negative', 'letters']
)
invalid_urban_categories['percentage_%'] = (
    invalid_urban_categories['count'] / total_rows * 100
).round(2)

print('\nInvalid UrbanRural categories (empty, 0, negative, letters):')
display(invalid_urban_categories)

# Duplicate values breakdown
duplicate_urban_values = (
    urban_std[duplicate_mask_urban]
    .value_counts(dropna=False)
    .rename_axis('UrbanRural_value')
    .to_frame('count')
)
if len(duplicate_urban_values) > 0:
    duplicate_urban_values['percentage_%'] = (
        duplicate_urban_values['count'] / total_rows * 100
    ).round(2)

print('\nDuplicated UrbanRural values:')
display(duplicate_urban_values)

print('\nSample invalid UrbanRural rows (up to 20):')
display(df.loc[invalid_mask_urban, ['UrbanRural', 'State', 'Accept']].head(20))

print('\nSample duplicate UrbanRural rows (up to 20):')
display(df.loc[duplicate_mask_urban, ['UrbanRural', 'State', 'Accept']].head(20))

UrbanRural data quality summary:


,count,percentage_%
total_rows,20768,100.00
missing,0,0.00
invalid_empty,0,0.00
invalid_0,6816,32.82
invalid_negative,0,0.00
invalid_letters,0,0.00
invalid_other_non_012,0,0.00
invalid_total,0,0.00
valid_0_1_2,20768,100.00
duplicate_rows,20768,100.00



UrbanRural value distribution (all categories):


,count,percentage_%
UrbanRural_value,,
1,12436,59.88
0,6816,32.82
2,1516,7.3



Invalid UrbanRural categories (empty, 0, negative, letters):


...

,count,percentage_%
UrbanRural_value,,
1,12436,59.88
0,6816,32.82
2,1516,7.3



Sample invalid UrbanRural rows (up to 20):


,UrbanRural,State,Accept



Sample duplicate UrbanRural rows (up to 20):


,UrbanRural,State,Accept
0,0,IL,0
1,1,IL,1
2,0,IL,1
3,1,IL,1
4,1,IL,0
5,1,IL,0
6,1,IL,1
7,0,IL,1
8,0,IL,1
9,1,IL,1


# RevLineCr
Text 	Indicates if this loan is a revolving line of credit. 	Y = Yes, N = No (may include non-standard values).

In [20]:
# RevLineCr audit: counts, percentages, missing values, and invalid categories
# Accepted values: Y, N
revline_raw = df['RevLineCr']

# Standardize to string for consistent category checks
revline_std = revline_raw.astype('string').str.strip().str.upper()

# Empty strings (explicit invalid category requested)
empty_mask_revline = revline_std.fillna('').eq('')

# Missing includes true NaN and blank strings
missing_mask_revline = revline_raw.isna() | empty_mask_revline

# Numeric conversion for numeric validity checks
revline_num = pd.to_numeric(revline_std, errors='coerce')

# Invalid categories requested
zero_mask_revline = (~missing_mask_revline) & revline_num.eq(0)
negative_mask_revline = (~missing_mask_revline) & revline_num.lt(0)
letters_mask_revline = (~missing_mask_revline) & revline_std.str.contains(r'[A-Z]', na=False)

# Accepted values are only Y and N
valid_values_revline = {'Y', 'N'}
valid_mask_revline = (~missing_mask_revline) & revline_std.isin(valid_values_revline)

# Different letters: alphabetic values that are not Y or N
different_letters_mask_revline = letters_mask_revline & (~revline_std.isin(valid_values_revline))

# Invalid mask: non-missing values not in accepted values
invalid_mask_revline = (~missing_mask_revline) & (~valid_mask_revline)

# Optional catch-all for invalid values not covered by requested categories
other_invalid_mask_revline = invalid_mask_revline & (
    ~zero_mask_revline
    & ~negative_mask_revline
    & ~different_letters_mask_revline
)

total_rows = len(df)
summary_revline = pd.DataFrame(
    {'count': [
        total_rows,
        int(missing_mask_revline.sum()),
        int(empty_mask_revline.sum()),
        int(zero_mask_revline.sum()),
        int(negative_mask_revline.sum()),
        int(different_letters_mask_revline.sum()),
        int(other_invalid_mask_revline.sum()),
        int(invalid_mask_revline.sum()),
        int(valid_mask_revline.sum())
    ]},
    index=[
        'total_rows',
        'missing',
        'invalid_empty',
        'invalid_0',
        'invalid_negative',
        'invalid_different_letters_not_YN',
        'invalid_other_non_YN',
        'invalid_total',
        'valid_YN'
    ]
)
summary_revline['percentage_%'] = (summary_revline['count'] / total_rows * 100).round(2)

print('RevLineCr data quality summary:')
display(summary_revline)

# Full distribution (including missing shown as MISSING)
revline_distribution = (
    revline_std.fillna('MISSING')
    .replace({'': 'MISSING'})
    .value_counts(dropna=False)
    .rename_axis('RevLineCr_value')
    .to_frame('count')
)
revline_distribution['percentage_%'] = (revline_distribution['count'] / total_rows * 100).round(2)

print('\nRevLineCr value distribution (all categories):')
display(revline_distribution)

# Explicit invalid category table requested
invalid_revline_categories = pd.DataFrame(
    {'count': [
        int(empty_mask_revline.sum()),
        int(zero_mask_revline.sum()),
        int(negative_mask_revline.sum()),
        int(different_letters_mask_revline.sum())
    ]},
    index=['empty', '0', 'negative', 'different_letters_not_YN']
)
invalid_revline_categories['percentage_%'] = (
    invalid_revline_categories['count'] / total_rows * 100
).round(2)

print('\nInvalid RevLineCr categories (empty, 0, negative, different letters):')
display(invalid_revline_categories)

print('\nSample invalid RevLineCr rows (up to 20):')
display(df.loc[invalid_mask_revline, ['RevLineCr', 'LowDoc', 'Accept']].head(20))

RevLineCr data quality summary:


,count,percentage_%
total_rows,20768,100.00
missing,126,0.61
invalid_empty,126,0.61
invalid_0,5046,24.30
invalid_negative,0,0.00
invalid_different_letters_not_YN,282,1.36
invalid_other_non_YN,0,0.00
invalid_total,5328,25.65
valid_YN,15314,73.74



RevLineCr value distribution (all categories):


,count,percentage_%
RevLineCr_value,,
N,10037,48.33
Y,5277,25.41
0,5046,24.3
T,281,1.35
MISSING,126,0.61
Q,1,0.0



Invalid RevLineCr categories (empty, 0, negative, different letters):


,count,percentage_%
empty,126,0.61
0,5046,24.30
negative,0,0.00
different_letters_not_YN,282,1.36



Sample invalid RevLineCr rows (up to 20):


,RevLineCr,LowDoc,Accept
5,0,Y,0
9,0,N,1
11,0,N,1
13,0,N,1
16,0,N,1
17,0,N,1
24,0,Y,0
25,0,N,1
28,0,N,0
30,0,N,1


# LowDoc
Text 	Indicates if this is a low-documentation loan. 	Y = Yes, N = No (may include non-standard values).

In [21]:
# LowDoc audit: counts, percentages, missing values, and invalid categories
# Accepted values: Y, N
lowdoc_raw = df['LowDoc']

# Standardize to string for consistent category checks
lowdoc_std = lowdoc_raw.astype('string').str.strip().str.upper()

# Empty strings (explicit invalid category requested)
empty_mask_lowdoc = lowdoc_std.fillna('').eq('')

# Missing includes true NaN and blank strings
missing_mask_lowdoc = lowdoc_raw.isna() | empty_mask_lowdoc

# Numeric conversion for numeric validity checks
lowdoc_num = pd.to_numeric(lowdoc_std, errors='coerce')

# Invalid categories requested
zero_mask_lowdoc = (~missing_mask_lowdoc) & lowdoc_num.eq(0)
negative_mask_lowdoc = (~missing_mask_lowdoc) & lowdoc_num.lt(0)
letters_mask_lowdoc = (~missing_mask_lowdoc) & lowdoc_std.str.contains(r'[A-Z]', na=False)

# Accepted values are only Y and N
valid_values_lowdoc = {'Y', 'N'}
valid_mask_lowdoc = (~missing_mask_lowdoc) & lowdoc_std.isin(valid_values_lowdoc)

# Different letters: alphabetic values that are not Y or N
different_letters_mask_lowdoc = letters_mask_lowdoc & (~lowdoc_std.isin(valid_values_lowdoc))

# Invalid mask: non-missing values not in accepted values
invalid_mask_lowdoc = (~missing_mask_lowdoc) & (~valid_mask_lowdoc)

# Optional catch-all for invalid values not covered by requested categories
other_invalid_mask_lowdoc = invalid_mask_lowdoc & (
    ~zero_mask_lowdoc
    & ~negative_mask_lowdoc
    & ~different_letters_mask_lowdoc
)

total_rows = len(df)
summary_lowdoc = pd.DataFrame(
    {'count': [
        total_rows,
        int(missing_mask_lowdoc.sum()),
        int(empty_mask_lowdoc.sum()),
        int(zero_mask_lowdoc.sum()),
        int(negative_mask_lowdoc.sum()),
        int(different_letters_mask_lowdoc.sum()),
        int(other_invalid_mask_lowdoc.sum()),
        int(invalid_mask_lowdoc.sum()),
        int(valid_mask_lowdoc.sum())
    ]},
    index=[
        'total_rows',
        'missing',
        'invalid_empty',
        'invalid_0',
        'invalid_negative',
        'invalid_different_letters_not_YN',
        'invalid_other_non_YN',
        'invalid_total',
        'valid_YN'
    ]
)
summary_lowdoc['percentage_%'] = (summary_lowdoc['count'] / total_rows * 100).round(2)

print('LowDoc data quality summary:')
display(summary_lowdoc)

# Full distribution (including missing shown as MISSING)
lowdoc_distribution = (
    lowdoc_std.fillna('MISSING')
    .replace({'': 'MISSING'})
    .value_counts(dropna=False)
    .rename_axis('LowDoc_value')
    .to_frame('count')
)
lowdoc_distribution['percentage_%'] = (lowdoc_distribution['count'] / total_rows * 100).round(2)

print('\nLowDoc value distribution (all categories):')
display(lowdoc_distribution)

# Explicit invalid category table requested
invalid_lowdoc_categories = pd.DataFrame(
    {'count': [
        int(empty_mask_lowdoc.sum()),
        int(zero_mask_lowdoc.sum()),
        int(negative_mask_lowdoc.sum()),
        int(different_letters_mask_lowdoc.sum())
    ]},
    index=['empty', '0', 'negative', 'different_letters_not_YN']
)
invalid_lowdoc_categories['percentage_%'] = (
    invalid_lowdoc_categories['count'] / total_rows * 100
).round(2)

print('\nInvalid LowDoc categories (empty, 0, negative, different letters):')
display(invalid_lowdoc_categories)

print('\nSample invalid LowDoc rows (up to 20):')
display(df.loc[invalid_mask_lowdoc, ['LowDoc', 'RevLineCr', 'Accept']].head(20))

LowDoc data quality summary:


,count,percentage_%
total_rows,20768,100.00
missing,36,0.17
invalid_empty,36,0.17
invalid_0,26,0.13
invalid_negative,0,0.00
invalid_different_letters_not_YN,25,0.12
invalid_other_non_YN,0,0.00
invalid_total,51,0.25
valid_YN,20681,99.58



LowDoc value distribution (all categories):


,count,percentage_%
LowDoc_value,,
N,16903,81.39
Y,3778,18.19
MISSING,36,0.17
0,26,0.13
S,8,0.04
C,7,0.03
A,6,0.03
R,4,0.02



Invalid LowDoc categories (empty, 0, negative, different letters):


,count,percentage_%
empty,36,0.17
0,26,0.13
negative,0,0.00
different_letters_not_YN,25,0.12



Sample invalid LowDoc rows (up to 20):


,LowDoc,RevLineCr,Accept
56,S,N,1
394,S,N,1
412,0,NaN,1
1413,R,N,0
1949,0,NaN,1
1959,S,N,1
2225,C,N,1
2555,0,N,1
2994,0,N,1
3779,A,Y,1


# DisbursementDate
Date/Time 	Date when funds were disbursed to the borrower. 	Useful for time-based features.

In [24]:
# DisbursementDate audit: counts, percentages, missing values, and invalid categories
# Invalid categories requested: empty, 0, negative, non dates
disb_raw = df['DisbursementDate']

# Standardize to string for consistent category checks
disb_std = disb_raw.astype('string').str.strip().str.upper()

# Empty strings (explicit invalid category requested)
empty_mask_disb = disb_std.fillna('').eq('')

# Missing includes true NaN and blank strings
missing_mask_disb = disb_raw.isna() | empty_mask_disb

# Numeric conversion for 0/negative category checks
disb_num = pd.to_numeric(disb_std, errors='coerce')

# Parse dates to identify valid and non-date values
disb_parsed = pd.to_datetime(disb_std, errors='coerce')

# Invalid categories requested
zero_mask_disb = (~missing_mask_disb) & disb_num.eq(0)
negative_mask_disb = (~missing_mask_disb) & disb_num.lt(0)
non_date_mask_disb = (~missing_mask_disb) & disb_parsed.isna()

# Invalid mask as requested
invalid_mask_disb = (
    empty_mask_disb
    | zero_mask_disb
    | negative_mask_disb
    | non_date_mask_disb
)

# Valid DisbursementDate: non-missing values parseable as date and not numeric 0/negative
valid_mask_disb = (
    (~missing_mask_disb)
    & disb_parsed.notna()
    & (~zero_mask_disb)
    & (~negative_mask_disb)
)

total_rows = len(df)
summary_disb = pd.DataFrame(
    {'count': [
        total_rows,
        int(missing_mask_disb.sum()),
        int(empty_mask_disb.sum()),
        int(zero_mask_disb.sum()),
        int(negative_mask_disb.sum()),
        int(non_date_mask_disb.sum()),
        int(invalid_mask_disb.sum()),
        int(valid_mask_disb.sum())
    ]},
    index=[
        'total_rows',
        'missing',
        'invalid_empty',
        'invalid_0',
        'invalid_negative',
        'invalid_non_dates',
        'invalid_total',
        'valid_date'
    ]
)
summary_disb['percentage_%'] = (summary_disb['count'] / total_rows * 100).round(2)

print('DisbursementDate data quality summary:')
display(summary_disb)

# Full distribution (including missing shown as MISSING)
disb_distribution = (
    disb_std.fillna('MISSING')
    .replace({'': 'MISSING'})
    .value_counts(dropna=False)
    .rename_axis('DisbursementDate_value')
    .to_frame('count')
)
disb_distribution['percentage_%'] = (disb_distribution['count'] / total_rows * 100).round(2)

print('\nDisbursementDate value distribution (all categories):')
display(disb_distribution.head(30))

# Explicit invalid category table requested
invalid_disb_categories = pd.DataFrame(
    {'count': [
        int(empty_mask_disb.sum()),
        int(zero_mask_disb.sum()),
        int(negative_mask_disb.sum()),
        int(non_date_mask_disb.sum())
    ]},
    index=['empty', '0', 'negative', 'non_dates']
)
invalid_disb_categories['percentage_%'] = (
    invalid_disb_categories['count'] / total_rows * 100
).round(2)

print('\nInvalid DisbursementDate categories (empty, 0, negative, non dates):')
display(invalid_disb_categories)

print('\nSample invalid DisbursementDate rows (up to 20):')
display(df.loc[invalid_mask_disb, ['DisbursementDate', 'ApprovalDate', 'Accept']].head(20))

DisbursementDate data quality summary:


C:\Users\nyliz\AppData\Local\Temp\ipykernel_21416\2990956638.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  disb_parsed = pd.to_datetime(disb_std, errors='coerce')


,count,percentage_%
total_rows,20768,100.0
missing,84,0.4
invalid_empty,84,0.4
invalid_0,0,0.0
invalid_negative,0,0.0
invalid_non_dates,0,0.0
invalid_total,84,0.4
valid_date,0,0.0



DisbursementDate value distribution (all categories):


,count,percentage_%
DisbursementDate_value,,
28-FEB-06,348,1.68
31-JUL-95,298,1.43
30-APR-95,283,1.36
31-JAN-95,280,1.35
30-APR-07,279,1.34
30-APR-96,253,1.22
31-OCT-07,243,1.17
31-OCT-06,230,1.11
31-OCT-95,230,1.11



Invalid DisbursementDate categories (empty, 0, negative, non dates):


,count,percentage_%
empty,84,0.4
0,0,0.0
negative,0,0.0
non_dates,0,0.0



Sample invalid DisbursementDate rows (up to 20):


,DisbursementDate,ApprovalDate,Accept
224,NaN,28-Sep-04,1
257,NaN,4-Jan-84,0
613,NaN,12-Jun-12,1
1038,NaN,30-Jan-96,1
1187,NaN,6-Jul-07,0
1724,NaN,24-Apr-06,1
2020,NaN,21-Dec-99,1
2042,NaN,3-Aug-10,1
2307,NaN,28-Jan-83,0
2486,NaN,11-Apr-85,0


# DisbursementGross
Currency 	Total loan amount disbursed. 	Convert currency strings to numeric values before modeling.

In [28]:
# DisbursementGross Data Quality Audit
gross_raw = df['DisbursementGross']
gross_std = gross_raw.astype('string').str.strip()
empty_mask_gross = gross_std.fillna('').eq('')
missing_mask_gross = gross_raw.isna() | empty_mask_gross

# Clean currency format: remove $, commas, spaces - keep only digits and decimal point
gross_cleaned = gross_std.str.replace('$', '').str.replace(',', '').str.strip()
gross_num = pd.to_numeric(gross_cleaned, errors='coerce')

# Define invalid categories
zero_mask_gross = (~missing_mask_gross) & (gross_num == 0)
negative_mask_gross = (~missing_mask_gross) & (gross_num < 0)
# Letters means A-Z (actual letters, not $ or other currency symbols)
letters_mask_gross = (~missing_mask_gross) & gross_cleaned.str.contains(r'[A-Z]', na=False, regex=True)

# Valid means: not missing, parsed successfully to number, and not zero/negative/letters
valid_mask_gross = (~missing_mask_gross) & (gross_num.notna()) & (~zero_mask_gross) & (~negative_mask_gross) & (~letters_mask_gross)
invalid_mask_gross = (~missing_mask_gross) & (~valid_mask_gross)
other_invalid_mask_gross = invalid_mask_gross & (~zero_mask_gross & ~negative_mask_gross & ~letters_mask_gross)

# Summary statistics
total_rows = len(df)
summary_gross = pd.DataFrame({
    'metric': ['Total Rows', 'Missing/Empty', 'Valid', 'Invalid (Total)', 
               '  Empty', '  Zero', '  Negative', '  With Letters', '  Other Invalid'],
    'count': [
        total_rows,
        missing_mask_gross.sum(),
        valid_mask_gross.sum(),
        invalid_mask_gross.sum(),
        empty_mask_gross.sum(),
        zero_mask_gross.sum(),
        negative_mask_gross.sum(),
        letters_mask_gross.sum(),
        other_invalid_mask_gross.sum()
    ]
})
summary_gross['percentage'] = (summary_gross['count'] / total_rows * 100).round(2)
print("=" * 80)
print("DISBURSEMENTGROSS - DATA QUALITY AUDIT")
print("=" * 80)
display(summary_gross)

# Full distribution of values
print("\n" + "=" * 80)
print("Full Value Distribution (All Unique Values)")
print("=" * 80)
gross_distribution = gross_num.value_counts().reset_index().rename(columns={'DisbursementGross': 'value'})
gross_distribution.columns = ['value', 'count']
gross_distribution = gross_distribution.sort_values('count', ascending=False)
print(f"Total unique values: {len(gross_distribution)}")
display(gross_distribution.head(50))

# Invalid categories breakdown
print("\n" + "=" * 80)
print("Invalid Values by Category")
print("=" * 80)
invalid_gross_categories = pd.DataFrame({
    'category': ['Empty/Missing', 'Zero', 'Negative', 'With Letters', 'Other Invalid'],
    'count': [
        empty_mask_gross.sum(),
        zero_mask_gross.sum(),
        negative_mask_gross.sum(),
        letters_mask_gross.sum(),
        other_invalid_mask_gross.sum()
    ]
})
invalid_gross_categories['percentage'] = (invalid_gross_categories['count'] / total_rows * 100).round(2)
display(invalid_gross_categories)

# Sample invalid rows
print("\n" + "=" * 80)
print("Sample Invalid Rows (up to 20)")
print("=" * 80)
sample_invalid_gross = df.loc[invalid_mask_gross, ['DisbursementGross', 'ApprovalFY', 'State', 'BankState']].head(20)
display(sample_invalid_gross)

DISBURSEMENTGROSS - DATA QUALITY AUDIT


,metric,count,percentage
0,Total Rows,20768,100.0
1,Missing/Empty,0,0.0
2,Valid,20767,100.0
3,Invalid (Total),1,0.0
4,Empty,0,0.0
5,Zero,1,0.0
6,Negative,0,0.0
7,With Letters,0,0.0
8,Other Invalid,0,0.0



Full Value Distribution (All Unique Values)
Total unique values: 6643


,value,count
0,100000.0,1027
1,50000.0,1003
2,25000.0,672
3,150000.0,590
4,5000.0,436
5,10000.0,399
6,20000.0,341
7,35000.0,336
8,15000.0,298
9,75000.0,293



Invalid Values by Category


,category,count,percentage
0,Empty/Missing,0,0.0
1,Zero,1,0.0
2,Negative,0,0.0
3,With Letters,0,0.0
4,Other Invalid,0,0.0



Sample Invalid Rows (up to 20)


,DisbursementGross,ApprovalFY,State,BankState
11414,$0.00,1993,IL,IL


# BalanceGross
Currency 	Remaining gross balance outstanding. 	Often zero in some subsets; still useful to inspect.

In [29]:
# BalanceGross Data Quality Audit
balance_raw = df['BalanceGross']
balance_std = balance_raw.astype('string').str.strip()
empty_mask_balance = balance_std.fillna('').eq('')
missing_mask_balance = balance_raw.isna() | empty_mask_balance

# Clean currency format: remove $, commas, spaces - keep only digits and decimal point
balance_cleaned = balance_std.str.replace('$', '').str.replace(',', '').str.strip()
balance_num = pd.to_numeric(balance_cleaned, errors='coerce')

# Define invalid categories
zero_mask_balance = (~missing_mask_balance) & (balance_num == 0)
negative_mask_balance = (~missing_mask_balance) & (balance_num < 0)
# Letters means A-Z (actual letters, not $ or other currency symbols)
letters_mask_balance = (~missing_mask_balance) & balance_cleaned.str.contains(r'[A-Z]', na=False, regex=True)

# Valid means: not missing, parsed successfully to number, and not zero/negative/letters
valid_mask_balance = (~missing_mask_balance) & (balance_num.notna()) & (~zero_mask_balance) & (~negative_mask_balance) & (~letters_mask_balance)
invalid_mask_balance = (~missing_mask_balance) & (~valid_mask_balance)
other_invalid_mask_balance = invalid_mask_balance & (~zero_mask_balance & ~negative_mask_balance & ~letters_mask_balance)

# Summary statistics
total_rows = len(df)
summary_balance = pd.DataFrame({
    'metric': ['Total Rows', 'Missing/Empty', 'Valid', 'Invalid (Total)', 
               '  Empty', '  Zero', '  Negative', '  With Letters', '  Other Invalid'],
    'count': [
        total_rows,
        missing_mask_balance.sum(),
        valid_mask_balance.sum(),
        invalid_mask_balance.sum(),
        empty_mask_balance.sum(),
        zero_mask_balance.sum(),
        negative_mask_balance.sum(),
        letters_mask_balance.sum(),
        other_invalid_mask_balance.sum()
    ]
})
summary_balance['percentage'] = (summary_balance['count'] / total_rows * 100).round(2)
print("=" * 80)
print("BALANCEGROSS - DATA QUALITY AUDIT")
print("=" * 80)
display(summary_balance)

# Full distribution of values
print("\n" + "=" * 80)
print("Full Value Distribution (All Unique Values)")
print("=" * 80)
balance_distribution = balance_num.value_counts().reset_index().rename(columns={'BalanceGross': 'value'})
balance_distribution.columns = ['value', 'count']
balance_distribution = balance_distribution.sort_values('count', ascending=False)
print(f"Total unique values: {len(balance_distribution)}")
display(balance_distribution.head(50))

# Invalid categories breakdown
print("\n" + "=" * 80)
print("Invalid Values by Category")
print("=" * 80)
invalid_balance_categories = pd.DataFrame({
    'category': ['Empty/Missing', 'Zero', 'Negative', 'With Letters', 'Other Invalid'],
    'count': [
        empty_mask_balance.sum(),
        zero_mask_balance.sum(),
        negative_mask_balance.sum(),
        letters_mask_balance.sum(),
        other_invalid_mask_balance.sum()
    ]
})
invalid_balance_categories['percentage'] = (invalid_balance_categories['count'] / total_rows * 100).round(2)
display(invalid_balance_categories)

# Sample invalid rows with Accept column for comparison
print("\n" + "=" * 80)
print("Sample Invalid Rows (up to 20) - Comparison with Accept Column")
print("=" * 80)
sample_invalid_balance = df.loc[invalid_mask_balance, ['BalanceGross', 'DisbursementGross', 'Accept']].head(20)
display(sample_invalid_balance)

BALANCEGROSS - DATA QUALITY AUDIT


,metric,count,percentage
0,Total Rows,20768,100.0
1,Missing/Empty,0,0.0
2,Valid,0,0.0
3,Invalid (Total),20768,100.0
4,Empty,0,0.0
5,Zero,20768,100.0
6,Negative,0,0.0
7,With Letters,0,0.0
8,Other Invalid,0,0.0



Full Value Distribution (All Unique Values)
Total unique values: 1


,value,count
0,0.0,20768



Invalid Values by Category


,category,count,percentage
0,Empty/Missing,0,0.0
1,Zero,20768,100.0
2,Negative,0,0.0
3,With Letters,0,0.0
4,Other Invalid,0,0.0



Sample Invalid Rows (up to 20) - Comparison with Accept Column


,BalanceGross,DisbursementGross,Accept
0,$0.00,"$600,000.00",0
1,$0.00,"$25,400.00",1
2,$0.00,"$20,000.00",1
3,$0.00,"$75,000.00",1
4,$0.00,"$50,000.00",0
5,$0.00,"$118,000.00",0
6,$0.00,"$50,000.00",1
7,$0.00,"$150,000.00",1
8,$0.00,"$1,000,000.00",1
9,$0.00,"$33,700.00",1


# Accept
Text 	Loan approval status; 0 = Not approved, 1 = Approved

In [30]:
# Accept Data Quality Audit
accept_raw = df['Accept']
accept_std = accept_raw.astype('string').str.strip().str.upper()

# Missing and empty
empty_mask_accept = accept_std.fillna('').eq('')
missing_mask_accept = accept_raw.isna() | empty_mask_accept

# Numeric and text checks
accept_num = pd.to_numeric(accept_std, errors='coerce')
negative_mask_accept = (~missing_mask_accept) & accept_num.lt(0)
letters_mask_accept = (~missing_mask_accept) & accept_std.str.contains(r'[A-Z]', na=False)

# Accepted values: 0 and 1 only
valid_values_accept = {'0', '1'}
valid_mask_accept = (~missing_mask_accept) & accept_std.isin(valid_values_accept)
invalid_mask_accept = (~missing_mask_accept) & (~valid_mask_accept)

# Invalid subtype: numeric but not 0/1 and not negative
non_binary_numeric_mask_accept = (
    (~missing_mask_accept)
    & accept_num.notna()
    & (~accept_std.isin(valid_values_accept))
    & (~negative_mask_accept)
)

# Summary statistics
total_rows = len(df)
summary_accept = pd.DataFrame({
    'metric': [
        'Total Rows', 'Missing/Empty', 'Valid (0 or 1)', 'Invalid (Total)',
        '  Empty', '  Negative', '  With Letters', '  Non-binary Numeric'
    ],
    'count': [
        total_rows,
        missing_mask_accept.sum(),
        valid_mask_accept.sum(),
        invalid_mask_accept.sum(),
        empty_mask_accept.sum(),
        negative_mask_accept.sum(),
        letters_mask_accept.sum(),
        non_binary_numeric_mask_accept.sum()
    ]
})
summary_accept['percentage'] = (summary_accept['count'] / total_rows * 100).round(2)

print('=' * 80)
print('ACCEPT - DATA QUALITY AUDIT')
print('=' * 80)
display(summary_accept)

# Full distribution of values
print('\n' + '=' * 80)
print('Full Value Distribution (All Unique Values)')
print('=' * 80)
accept_distribution = (
    accept_std.value_counts(dropna=False)
    .rename_axis('value')
    .reset_index(name='count')
)
accept_distribution['percentage'] = (accept_distribution['count'] / total_rows * 100).round(2)
print(f'Total unique values: {len(accept_distribution)}')
display(accept_distribution)

# Invalid categories breakdown
print('\n' + '=' * 80)
print('Invalid Values by Category')
print('=' * 80)
invalid_accept_categories = pd.DataFrame({
    'category': ['Empty/Missing', 'Negative', 'With Letters', 'Non-binary Numeric'],
    'count': [
        empty_mask_accept.sum(),
        negative_mask_accept.sum(),
        letters_mask_accept.sum(),
        non_binary_numeric_mask_accept.sum()
    ]
})
invalid_accept_categories['percentage'] = (invalid_accept_categories['count'] / total_rows * 100).round(2)
display(invalid_accept_categories)

# Sample invalid rows with comparison columns
print('\n' + '=' * 80)
print('Sample Invalid Rows (up to 20) - Comparison Columns')
print('=' * 80)
display(df.loc[invalid_mask_accept, ['Accept', 'DisbursementDate', 'ApprovalDate']].head(20))

ACCEPT - DATA QUALITY AUDIT


,metric,count,percentage
0,Total Rows,20768,100.0
1,Missing/Empty,0,0.0
2,Valid (0 or 1),20768,100.0
3,Invalid (Total),0,0.0
4,Empty,0,0.0
5,Negative,0,0.0
6,With Letters,0,0.0
7,Non-binary Numeric,0,0.0



Full Value Distribution (All Unique Values)
Total unique values: 2


,value,count,percentage
0,1,16019,77.13
1,0,4749,22.87



Invalid Values by Category


,category,count,percentage
0,Empty/Missing,0,0.0
1,Negative,0,0.0
2,With Letters,0,0.0
3,Non-binary Numeric,0,0.0



Sample Invalid Rows (up to 20) - Comparison Columns


,Accept,DisbursementDate,ApprovalDate
